# Projeto revisado — localização-alocação escolar em Lavras

**GCC118 · Programação Matemática · Universidade Federal de Lavras**

**Pergunta de pesquisa:** dada a distribuição espacial da população em idade escolar e a rede pública em funcionamento, como alocar estudantes às escolas existentes e identificar localizações candidatas para novas unidades, minimizando deslocamentos e respeitando restrições de capacidade?

Este é um estudo de **planejamento hipotético**, não uma recomendação administrativa de construir escolas. Capacidade operacional, participação na rede pública, disponibilidade de terrenos e custos reais não estão diretamente observados nas bases utilizadas.

Os notebooks são o **código-fonte principal**: a análise, a construção de instâncias e a modelagem ficam nos próprios documentos, sem depender de módulos Python adicionais.

## 1. Fluxo de trabalho

1. `01_demanda_ibge.ipynb`: população por faixa etária no setor censitário, auditoria e cenários de demanda independentes das matrículas.
2. `02_escolas_inep.ipynb`: escolas existentes, matrículas e turmas como indicadores distintos; cenários explícitos de capacidade operacional.
3. `03_instancias_candidatos.ipynb`: integração geográfica, geração **experimental** de locais candidatos e matrizes de distâncias.
4. `04_modelos_gurobi.ipynb`: programação inteira mista, teste sintético, diagnóstico de viabilidade, número mínimo de novas escolas e minimização de deslocamentos.
5. `05_resultados.ipynb`: indicadores de cobertura, capacidade, número de aberturas, distância, tempos e status.

Execute nesta ordem. A transferência entre notebooks usa CSV e JSON em `outputs/localizacao_escolar/lavras/` (pasta ignorada pelo Git). Cada notebook deve poder ser reexecutado isoladamente depois que os anteriores produzirem as saídas.

## 2. Distinção indispensável: dados versus hipóteses

| Elemento | Fonte / definição | Natureza |
|---|---|---|
| Geometria dos setores e população residente | IBGE, Censo 2022 | Observado |
| Faixas 5–9, 10–14, 15–19 anos | IBGE, Censo 2022 | Observado com supressões |
| Demanda por etapa | Grupos etários como **proxy**, com participação pública \(\rho\) | Cenário, não matrícula real |
| Escolas existentes, matrículas e turmas | INEP, Censo Escolar 2025 | Observado |
| Capacidade \(Q_j\) | Turmas × lotação hipotética por turma | Cenário, não vagas certificadas |
| Local de nova escola | Pontos representativos de setores selecionados | Candidato analítico; terreno não verificado |
| Capacidade de unidade nova | Número hipotético de turmas × alunos/turma | Cenário |
| Distância | Linha reta em projeção UTM local | Aproximação; não equivale ao trajeto viário |
| Custo de abertura | Não estimado | Substituído por **número máximo de aberturas** |

Nunca interpretar `rho=0.8` como taxa pública apurada no município. Os grupos etários não coincidem exatamente com a idade/série escolar e não mensuram diretamente quem mora em Lavras e frequenta escola pública.

## 3. Modelagem em duas decisões

Para cada etapa de ensino \(e\), sejam \(I\) os setores com demanda positiva, \(J\) as escolas existentes e \(K\) os candidatos de novas escolas. \(w_i\) é a demanda estimada do setor, \(q_j\) a capacidade hipotética existente, \(u_k\) a capacidade de uma escola nova e \(d_{ih}\) a distância do setor \(i\) ao local \(h\).

Decisões: \(z_{ih}\in\mathbb{Z}_{\ge0}\) estudantes enviados de \(i\) a \(h\), e \(y_k\in\{0,1\}\) indicando construção no candidato \(k\). A demanda **pode ser dividida** entre escolas, diferentemente da atribuição integral de setores da formulação anterior.

\[
\sum_{h\in J\cup K}z_{ih}=w_i \quad \forall i\in I
\]
\[
\sum_{i\in I}z_{ij}\le q_j\quad \forall j\in J
\]
\[
\sum_{i\in I}z_{ik}\le u_k y_k\quad \forall k\in K
\]

**Primeira etapa:** \(\min \sum_k y_k\) (quantidade mínima de escolas novas para atender toda a demanda sob as regras de distância, quando definidas).

**Segunda etapa:** fixado um limite \(\sum_k y_k\le p\), \(\min \sum_{i,h}d_{ih}z_{ih}\) (menor deslocamento total). Testar \(p=p_{\min},p_{\min}+1,p_{\min}+2\) e comparar fronteira de compromisso.

Uma distância máxima por par pode ser adicionada como restrição de elegibilidade, mas pode produzir inviabilidade. A versão básica aceita todos os pares; medir percursos acima de limites é obrigatório na análise.

## 4. Por que não usar matrículas como demanda?

Na implementação anterior, \(\sum_i w_i\) era calibrado para igualar \(\sum_j M_j\), e \(Q_j=\lceil\alpha M_j\rceil\). Isso torna o balanço de oferta/demanda parcialmente artificial. Aqui, as matrículas são **indicadores de uso observado** da unidade; a demanda é construída independentemente a partir de demografia.

A estimativa por faixa etária continua sujeita a limitações: nem toda criança está na rede pública, idade e etapa escolar não coincidem perfeitamente, e anos de referência são diferentes. Em vez de ocultar essas incertezas, variamos \(\rho\) e a lotação hipotética por turma.

## 5. Referências de método (SciSpace)

- Menezes & Pizzolato (2014), *Locating Public Schools in Fast Expanding Areas*: [estudo brasileiro de p-medianas capacitadas e cobertura](https://scispace.com/papers/locating-public-schools-in-fast-expanding-areas-application-8j9p2k9cle).
- Mattsson (1986), *Residential location and school planning*: [escolas existentes e novas instalações](https://scispace.com/papers/residential-location-and-school-planning-in-a-tightening-4jgwqv96x5).
- Delmelle et al. (2014), *A multi-period capacitated school location problem*: [capacidade, orçamento e deslocamento](https://scispace.com/papers/a-multi-period-capacitated-school-location-problem-with-2y0qyudjl3).
- Chen, Thill & Delmelle (2018), *iGLASS*: [restrições de capacidade e atribuições distantes](https://scispace.com/papers/iglass-an-open-source-sdss-for-public-school-location-33ig1f9tfd).

O modelo desenvolvido nos notebooks é uma **formulação didática inspirada** nesses trabalhos; não reproduz integralmente o modelo nem os resultados dos artigos.

In [ ]:
from pathlib import Path
import pandas as pd

raiz = Path.cwd().resolve()
if not (raiz / "pyproject.toml").exists():
    for pai in raiz.parents:
        if (pai / "pyproject.toml").exists():
            raiz = pai
            break
assert (raiz / "pyproject.toml").exists(), "Execute no projeto pm."
saida = raiz / "outputs" / "localizacao_escolar" / "lavras"
saida.mkdir(parents=True, exist_ok=True)
print("Raiz:", raiz)
print("Saídas:", saida)
print("Próximo notebook: 01_demanda_ibge.ipynb")